# In-Class Assessment Solution: Comparing Classification Algorithms

## Objective
This notebook provides a complete solution for the student performance classification task using:

- Logistic Regression
- Decision Tree
- Neural Network

The goal is to predict whether a student will:

- **1 = Pass**
- **0 = Need Support**

In [ ]:
# ============================================================
# STEP 1: Import the required libraries
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
# ============================================================
# STEP 2: Generate the synthetic dataset
# ============================================================

np.random.seed(42)

# Number of students
n = 400

# Generate feature columns
study_hours_per_week = np.random.randint(0, 26, n)
attendance_rate = np.random.randint(40, 101, n)
assignment_avg = np.random.randint(35, 101, n)
quiz_avg = np.random.randint(30, 101, n)
lab_performance = np.random.randint(30, 101, n)
participation_score = np.random.randint(0, 11, n)
previous_gpa = np.round(np.random.uniform(1.5, 4.0, n), 2)

# Create a weighted score with some noise
performance_score = (
    0.10 * study_hours_per_week +
    0.20 * attendance_rate +
    0.20 * assignment_avg +
    0.15 * quiz_avg +
    0.15 * lab_performance +
    1.5 * participation_score +
    8 * previous_gpa +
    np.random.normal(0, 8, n)
)

# Create binary target
final_outcome = (performance_score > np.median(performance_score)).astype(int)

# Create DataFrame
df = pd.DataFrame({
    "study_hours_per_week": study_hours_per_week,
    "attendance_rate": attendance_rate,
    "assignment_avg": assignment_avg,
    "quiz_avg": quiz_avg,
    "lab_performance": lab_performance,
    "participation_score": participation_score,
    "previous_gpa": previous_gpa,
    "final_outcome": final_outcome
})

df.head()

In [ ]:
# ============================================================
# STEP 3: Explore the dataset
# ============================================================

print("Shape of dataset:", df.shape)
print("\nDataset information:")
df.info()

print("\nSummary statistics:")
display(df.describe())

print("\nClass distribution:")
print(df["final_outcome"].value_counts())

## Reflection Question 1

A reasonable expectation is that `attendance_rate`, `assignment_avg`, `quiz_avg`, `lab_performance`, and `previous_gpa` will be among the strongest predictors because they are directly tied to academic performance. The three models may not perform exactly the same because they learn patterns differently. Logistic regression assumes a relatively smooth linear decision boundary, decision trees learn rule-based splits, and neural networks can capture more complex interactions.

In [ ]:
# ============================================================
# STEP 4: Define features (X) and target (y)
# ============================================================

X = df.drop("final_outcome", axis=1)
y = df["final_outcome"]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

In [ ]:
# ============================================================
# STEP 5: Split the data into training and testing sets
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
# ============================================================
# STEP 6: Standardize the features
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training feature matrix shape:", X_train_scaled.shape)
print("Scaled testing feature matrix shape:", X_test_scaled.shape)

# Model 1: Logistic Regression

In [ ]:
# ============================================================
# STEP 7: Train a Logistic Regression model
# ============================================================

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)

y_pred_log = log_model.predict(X_test_scaled)

In [ ]:
# ============================================================
# STEP 8: Evaluate Logistic Regression
# ============================================================

log_report_text = classification_report(y_test, y_pred_log)
log_accuracy = accuracy_score(y_test, y_pred_log)

print("Logistic Regression Classification Report")
print(log_report_text)
print("Accuracy:", round(log_accuracy, 4))

# Model 2: Decision Tree

In [ ]:
# ============================================================
# STEP 9: Train a Decision Tree model
# ============================================================

tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train, y_train)

y_pred_tree = tree_model.predict(X_test)

In [ ]:
# ============================================================
# STEP 10: Evaluate Decision Tree
# ============================================================

tree_report_text = classification_report(y_test, y_pred_tree)
tree_accuracy = accuracy_score(y_test, y_pred_tree)

print("Decision Tree Classification Report")
print(tree_report_text)
print("Accuracy:", round(tree_accuracy, 4))

# Model 3: Neural Network

In [ ]:
# ============================================================
# STEP 11: Train a Neural Network model
# ============================================================

nn_model = MLPClassifier(hidden_layer_sizes=(10, 5), max_iter=1000, random_state=42)
nn_model.fit(X_train_scaled, y_train)

y_pred_nn = nn_model.predict(X_test_scaled)

In [ ]:
# ============================================================
# STEP 12: Evaluate Neural Network
# ============================================================

nn_report_text = classification_report(y_test, y_pred_nn)
nn_accuracy = accuracy_score(y_test, y_pred_nn)

print("Neural Network Classification Report")
print(nn_report_text)
print("Accuracy:", round(nn_accuracy, 4))

In [ ]:
# ============================================================
# STEP 13: Create a comparison table
# ============================================================

log_report = classification_report(y_test, y_pred_log, output_dict=True)
tree_report = classification_report(y_test, y_pred_tree, output_dict=True)
nn_report = classification_report(y_test, y_pred_nn, output_dict=True)

comparison_df = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": log_accuracy,
        "Precision": log_report["1"]["precision"],
        "Recall": log_report["1"]["recall"],
        "F1-Score": log_report["1"]["f1-score"]
    },
    {
        "Model": "Decision Tree",
        "Accuracy": tree_accuracy,
        "Precision": tree_report["1"]["precision"],
        "Recall": tree_report["1"]["recall"],
        "F1-Score": tree_report["1"]["f1-score"]
    },
    {
        "Model": "Neural Network",
        "Accuracy": nn_accuracy,
        "Precision": nn_report["1"]["precision"],
        "Recall": nn_report["1"]["recall"],
        "F1-Score": nn_report["1"]["f1-score"]
    }
])

In [ ]:
# ============================================================
# STEP 14: Display the comparison table
# ============================================================

comparison_df.round(3)

## Reflection Question 2

The best-performing model may vary slightly because of the random train-test split and the noise in the synthetic data. On this dataset, logistic regression and the neural network often perform strongly because the target was generated from a weighted combination of features. The decision tree may still perform reasonably well, but it can overfit more easily. If interpretability is important, logistic regression or decision tree would usually be preferred over a neural network because their logic is easier to explain.